# LA 01 — Binance data readiness audit

Stage A, independent Liquidity Absorption branch. **Read-only sources; no signal, model, or P&L.** Probe days are selected by fixed chronology (first, midpoint, last), not outcomes. Outputs are written as CSV/JSON under `01/`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

p = Path.cwd().resolve()
while not (p / "src" / "data_engineering").exists():
    if p == p.parent:
        raise RuntimeError("Launch Jupyter from repository or subfolder")
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from src.data_engineering.modeling.liquidity_absorption_stage_a import VERSION, binance_root, run_audit
print("Repo:", p, "| Audit:", VERSION)

Repo: D:\Projects\deribit_market_data | Audit: LA-A01-0.1.0


In [2]:
DATA_ROOT = binance_root(p)
OUT = p / "notebooks" / "Liquidity_Absorption" / "01"
print("Binance root:", DATA_ROOT)
print("Output:", OUT)

Binance root: D:\MarketData\Binance
Output: D:\Projects\deribit_market_data\notebooks\Liquidity_Absorption\01


In [3]:
decision = run_audit(DATA_ROOT, OUT)
decision

MergeError: Merge keys are not unique in left dataset; not a one-to-one merge
Duplicates in left:
  minute_ms
   1782864
   1782864
   1782864
   1782864
   1782864 ...

In [ ]:
for filename in decision["files"]:
    print("\n==", filename)
    f = OUT / filename
    if f.exists() and f.stat().st_size:
        display(pd.read_csv(f).head(12))
    else:
        print("Empty file; data not available")

## Interpretation checklist

- Are BTC and ETH perpetual minute bars present, monotone and free of duplicate open times?
- How many missing one-minute intervals occur within their historical spans?
- Are Binance spot 1m data available for the same assets?
- Are aggTrade files contiguous between the first and last day?
- Are sampled individual trades timestamp-ordered with valid price, quantity, aggressor side?
- Is the price/flow reconciliation broadly plausible? **This is descriptive; no statistical gate has been pre-registered for the relative error.**
- Do not interpret a top-volume bin as a predictive support/resistance zone. This is merely an audit visualization.

Commit `01/` outputs and the executed notebook to GitHub. Next: specify LA 02 ex-ante event definitions before screening outcomes.